In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# ==========================================
# 1. LOAD THE DATASETS FROM KAGGLE
# ==========================================

# NOTE: Paste your Kaggle Input directory paths here once you attach the dataset
# Example Kaggle path structure: "/kaggle/input/waiter-robot-sensor-logs/robot_data.csv"
robot_data_path = ""

# Read the CSV dataset
df = pd.read_csv(robot_data_path, low_memory=False)

print(f"Dataset Loaded Successfully from Kaggle! Matrix Shape: {df.shape}")


# ==========================================
# 2. DATASET FEATURES & TARGET EXTRACTION
# ==========================================

# Features based on the Arduino Waiter Robot Hardware Profile:
# - Current_State: 0=WAIT_INPUT, 1=GO_TO_TABLE, 2=RETURNING
# - IR_Left / IR_Right: 0=White Surface, 1=Black Line/Node
# - Distance_CM: Ultrasonic distance reading
# - Target_Table: Selected Table from 2x3 Keypad Matrix (1-5)

X = df[['Current_State', 'IR_Left', 'IR_Right', 'Distance_CM', 'Target_Table']]

# Target Label (Predicted_Action): 
# 0=WAIT_KEYPAD, 1=FORWARD, 2=LEFT, 3=RIGHT, 4=DELIVERING_COUNTDOWN, 5=STOP_OBSTACLE
y = df['Predicted_Action']


# ==========================================
# 3. TRAIN-TEST SPLIT & MODEL TRAINING
# ==========================================

# Split dataset into training and validation sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# Train a small, lightweight RandomForest perfect for embedded logic emulation
model = RandomForestClassifier(n_estimators=15, random_state=42)
model.fit(X_train, y_train)


# ==========================================
# 4. EVALUATE EMBEDDED MODEL ACCURACY
# ==========================================

y_pred = model.predict(X_test)
print(f"\n[MODEL EVALUATION] Emulation Accuracy Score: {accuracy_score(y_test, y_pred) * 100}%")


# ==========================================
# 5. DEFINE REAL-TIME ARDUINO ACTION FUNCTION
# ==========================================

def get_waiter_robot_action(state, left, right, distance, target):
    """
    Takes live sensor framework inputs and predicts the target hardware state 
    along with corresponding LCD message and RGB indicator statuses.
    """
    input_data = np.array([[state, left, right, distance, target]])
    prediction = model.predict(input_data)[0]
    
    # State mapping mirror from the original I2C LCD messages and RGB module
    action_map = {
        0: {"LCD": "Press 1-5 Table", "RGB": "RED=OFF, GREEN=OFF", "Motors": "STOPPED"},
        1: {"LCD": "Moving Forward", "RGB": "RED=OFF, GREEN=ON", "Motors": "FORWARD (105, 105)"},
        2: {"LCD": "Adjusting Left", "RGB": "RED=OFF, GREEN=ON", "Motors": "TURN LEFT (-105, 105)"},
        3: {"LCD": "Adjusting Right", "RGB": "RED=OFF, GREEN=ON", "Motors": "TURN RIGHT (105, -105)"},
        4: {"LCD": "Delivering... Wait", "RGB": "RED=ON, GREEN=ON", "Motors": "STOPPED (Countdown)"},
        5: {"LCD": "Obstacle!", "RGB": "RED=ON, GREEN=OFF", "Motors": "STOPPED (Distance <= 7cm)"}
    }
    
    return action_map.get(prediction, {"LCD": "Unknown Error", "RGB": "UNKNOWN", "Motors": "STOPPED"})


# ==========================================
# 6. RUN SIMULATION TEST CASES
# ==========================================

print("\n" + "="*50 + "\nRUNNING ROBOT STATE MACHINE SIMULATIONS:\n" + "="*50)

# Test 1: Robot is in GO_TO_TABLE state, clear line ahead, no obstacles (45cm)
# State=1 (GO_TO_TABLE), Left=0, Right=0, Distance=45, Target=3
test_1 = get_waiter_robot_action(state=1, left=0, right=0, distance=45, target=3)
print(f"[TEST 1 - Moving]:\n📺 LCD: {test_1['LCD']}\n🚨 RGB: {test_1['RGB']}\n⚙️ Motors: {test_1['Motors']}\n")

# Test 2: Robot encounters an obstacle closer than 7cm (e.g., 5cm)
test_2 = get_waiter_robot_action(state=1, left=0, right=0, distance=5, target=3)
print(f"[TEST 2 - Obstacle Check]:\n📺 LCD: {test_2['LCD']}\n🚨 RGB: {test_2['RGB']}\n⚙️ Motors: {test_2['Motors']}\n")


# ==========================================
# 7. INSTALL & LAUNCH GRADIO UI INTERFACE
# ==========================================

# Install gradio directly inside the Kaggle notebook instance
!pip install -q gradio

import gradio as gr

def gradio_interface_wrapper(state, left, right, distance, target):
    state_dict = {"WAIT_INPUT": 0, "GO_TO_TABLE": 1, "RETURNING": 2}
    ir_dict = {"White Surface (0)": 0, "Black Line (1)": 1}
    
    s_val = state_dict[state]
    l_val = ir_dict[left]
    r_val = ir_dict[right]
    
    output = get_waiter_robot_action(s_val, l_val, r_val, distance, target)
    return f"📺 [LCD DISPLAY]: {output['LCD']}\n\n🚨 [RGB FLAGS]: {output['RGB']}\n\n⚙️ [MOTOR DRIVES]: {output['Motors']}"

# Render the interactive panel layout
demo = gr.Interface(
    fn=gradio_interface_wrapper,
    inputs=[
        gr.Dropdown(choices=["WAIT_INPUT", "GO_TO_TABLE", "RETURNING"], value="GO_TO_TABLE", label="Current Robot State"),
        gr.Dropdown(choices=["White Surface (0)", "Black Line (1)"], value="White Surface (0)", label="Left IR Sensor State"),
        gr.Dropdown(choices=["White Surface (0)", "Black Line (1)"], value="White Surface (0)", label="Right IR Sensor State"),
        gr.Slider(minimum=0, maximum=100, value=50, label="Ultrasonic Range Sensor (CM)"),
        gr.Slider(minimum=1, maximum=5, value=3, step=1, label="Target Table (Keypad Input)")
    ],
    outputs=gr.Textbox(label="Predicted Hardware Event Logs:", lines=5),
    title="🤖 Smart Waiter Robot Emulator (Decision Trees)",
    description="Adjust simulated sensor profiles to observe how the Random Forest model predicts real-time physical actions based on your embedded C++ code architecture.",
    theme="soft"
)

demo.launch(share=True)
